# ShakeSense — Personalised Motor Model Training

This notebook trains **one personalised anomaly-detection profile for one physical motor** using its collected sensor data. It does **not** generate or use synthetic data.

**Workflow**
1. Set the device ID, raw-data file, sensor column names, sampling rate, and window size.
2. Load the ESP32-collected MPU6050 + DS18B20 time-series data.
3. Calculate vibration features inside this notebook for every window.
4. Select windows confirmed to represent healthy operation.
5. Train and calibrate a device-specific Isolation Forest.
6. Evaluate it if independently labelled test data is available, then save the profile.

> Important: training requires real collected data. The notebook cannot determine by itself whether a window is truly healthy. Mark/identify healthy calibration periods using maintenance checks or a known-good motor condition.

## 1. Install and import libraries

In [ ]:
%pip install -q numpy pandas scipy scikit-learn matplotlib joblib

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import kurtosis
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

RANDOM_STATE = 42

## 2. Configure this motor and its raw sensor data

Edit these variables. The example column names are placeholders: change them to match the CSV exported by your ESP32 logger.

Expected raw data: one row per sensor sample, with accelerometer readings and a temperature reading. A timestamp column is optional but recommended.

- Accelerometer values should be consistently scaled (for example, all in `g` or all in `m/s²`).
- If your logger stores MPU6050 raw ADC counts, convert them to acceleration units before feature extraction by setting `ACCEL_SCALE`.
- `WINDOW_SIZE` is the number of samples per analysis window.
- `HEALTHY_FLAG_COLUMN` is optional. If present, rows/windows with flag `1` are treated as confirmed healthy. If absent, you will manually select a healthy time range in the next section.

In [ ]:
# -------- USER CONFIGURATION --------
DEVICE_ID = "SHAKESENSE_MOTOR_001"
DEVICE_TYPE = "fan"                 # e.g. "fan", "mixer", "pump"

RAW_CSV_PATH = "motor_raw_sensor_data.csv"
OUTPUT_DIR = "device_profiles"

# Change these to match your CSV headers
TIMESTAMP_COLUMN = "timestamp"      # set to None if no timestamp column
ACCEL_X_COLUMN = "accel_x"
ACCEL_Y_COLUMN = "accel_y"
ACCEL_Z_COLUMN = "accel_z"
TEMPERATURE_COLUMN = "temperature_c"
HEALTHY_FLAG_COLUMN = None          # e.g. "verified_healthy"; set None if unavailable

# Signal processing
SAMPLING_RATE_HZ = 100              # actual ESP32 sampling rate
WINDOW_SIZE = 1000                  # samples per window
OVERLAP = 0                         # 0 = no overlap; e.g. 0.5 = 50% overlap
SIGNAL_MODE = "magnitude"           # "magnitude", "x", "y", or "z"

# Set to 1.0 if values already in g or m/s².
# For MPU6050 raw counts at ±2g, use 1/16384 to convert to g.
ACCEL_SCALE = 1.0

# Personalised model settings
N_ESTIMATORS = 200
HEALTHY_THRESHOLD_QUANTILE = 0.01   # lower 1% tail of held-out healthy scores
CONTAMINATION = "auto"

# If there is no healthy flag column, define the healthy portion of the recording.
# Use sample indices, not seconds. Set both to None to use all data only if
# the ENTIRE recording is independently known to be healthy.
HEALTHY_START_SAMPLE = None
HEALTHY_END_SAMPLE = None

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Device:", DEVICE_ID, "| Type:", DEVICE_TYPE)

## 3. Load and validate the real collected data

In [ ]:
if not os.path.exists(RAW_CSV_PATH):
    raise FileNotFoundError(
        f"Could not find {RAW_CSV_PATH}. Upload/copy your real ESP32 CSV and update RAW_CSV_PATH."
    )

raw = pd.read_csv(RAW_CSV_PATH)
print("Raw rows:", len(raw))
print("Columns:", raw.columns.tolist())
display(raw.head())

required = [ACCEL_X_COLUMN, ACCEL_Y_COLUMN, ACCEL_Z_COLUMN, TEMPERATURE_COLUMN]
missing = [c for c in required if c not in raw.columns]
if missing:
    raise ValueError(f"Missing configured sensor columns: {missing}")

if TIMESTAMP_COLUMN is not None and TIMESTAMP_COLUMN not in raw.columns:
    print(f"Warning: timestamp column '{TIMESTAMP_COLUMN}' not found; sample indices will be used.")

for c in required:
    raw[c] = pd.to_numeric(raw[c], errors="coerce")

raw = raw.dropna(subset=required).reset_index(drop=True)
if len(raw) < WINDOW_SIZE:
    raise ValueError(f"Only {len(raw)} valid samples; need at least WINDOW_SIZE={WINDOW_SIZE}.")
print("Valid rows after cleaning:", len(raw))

## 4. Build the vibration signal and calculate features per window

For each window this calculates:
- mean
- RMS
- peak-to-peak
- kurtosis (Pearson definition)
- crest factor = peak absolute amplitude / RMS
- mean temperature for the same window

`magnitude` combines the three accelerometer axes as √(x² + y² + z²). If you use magnitude, the constant gravity component may influence mean/RMS. Keep the same signal definition, sensor mounting, units, sampling rate, and window length between calibration and live inference.

In [ ]:
def select_signal(frame):
    x = frame[ACCEL_X_COLUMN].to_numpy(dtype=float) * ACCEL_SCALE
    y = frame[ACCEL_Y_COLUMN].to_numpy(dtype=float) * ACCEL_SCALE
    z = frame[ACCEL_Z_COLUMN].to_numpy(dtype=float) * ACCEL_SCALE

    if SIGNAL_MODE == "x":
        return x
    if SIGNAL_MODE == "y":
        return y
    if SIGNAL_MODE == "z":
        return z
    if SIGNAL_MODE == "magnitude":
        return np.sqrt(x*x + y*y + z*z)
    raise ValueError("SIGNAL_MODE must be 'magnitude', 'x', 'y', or 'z'.")

def calculate_window_features(signal, temperature):
    signal = np.asarray(signal, dtype=float)
    rms = float(np.sqrt(np.mean(signal ** 2)))
    peak_abs = float(np.max(np.abs(signal)))
    return {
        "mean": float(np.mean(signal)),
        "rms": rms,
        "peak_to_peak": float(np.ptp(signal)),
        "kurtosis": float(kurtosis(signal, fisher=False, bias=False)) if len(signal) > 3 else np.nan,
        "crest_factor": float(peak_abs / rms) if rms > 0 else 0.0,
        "temperature_c": float(np.mean(temperature))
    }

signal = select_signal(raw)
temperature = raw[TEMPERATURE_COLUMN].to_numpy(dtype=float)

step = int(WINDOW_SIZE * (1 - OVERLAP))
if step < 1:
    raise ValueError("OVERLAP must be less than 1.0")

feature_rows = []
for start in range(0, len(raw) - WINDOW_SIZE + 1, step):
    end = start + WINDOW_SIZE
    feats = calculate_window_features(signal[start:end], temperature[start:end])
    feats["start_sample"] = start
    feats["end_sample"] = end
    feats["start_time_s"] = start / SAMPLING_RATE_HZ
    feats["end_time_s"] = end / SAMPLING_RATE_HZ
    feats["device_id"] = DEVICE_ID
    feature_rows.append(feats)

features = pd.DataFrame(feature_rows)
if features.empty:
    raise ValueError("No complete windows were produced. Check WINDOW_SIZE and data length.")

FEATURE_COLUMNS = ["mean", "rms", "peak_to_peak", "kurtosis", "crest_factor", "temperature_c"]
features = features.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURE_COLUMNS).reset_index(drop=True)

print("Feature windows:", len(features))
display(features.head())
display(features[FEATURE_COLUMNS].describe())

## 5. Identify confirmed healthy windows

Do not label data healthy just because the motor was running. Use a period when the motor is known to be operating normally.

**Option A:** If your CSV has a healthy flag column, set `HEALTHY_FLAG_COLUMN` above (1 = verified healthy).

**Option B:** Otherwise set `HEALTHY_START_SAMPLE` and `HEALTHY_END_SAMPLE` above to select the known-healthy sample-index range. If both are left `None`, the notebook stops rather than silently treating all readings as healthy.

In [ ]:
if HEALTHY_FLAG_COLUMN is not None:
    if HEALTHY_FLAG_COLUMN not in raw.columns:
        raise ValueError(f"Healthy flag column '{HEALTHY_FLAG_COLUMN}' not found.")
    # A window is considered healthy only if every raw sample in it is flagged 1.
    healthy_flags = pd.to_numeric(raw[HEALTHY_FLAG_COLUMN], errors="coerce").fillna(0).to_numpy()
    window_healthy = []
    for _, row in features.iterrows():
        s, e = int(row["start_sample"]), int(row["end_sample"])
        window_healthy.append(bool(np.all(healthy_flags[s:e] == 1)))
    features["verified_healthy"] = window_healthy

elif HEALTHY_START_SAMPLE is not None and HEALTHY_END_SAMPLE is not None:
    features["verified_healthy"] = (
        (features["start_sample"] >= HEALTHY_START_SAMPLE) &
        (features["end_sample"] <= HEALTHY_END_SAMPLE)
    )
else:
    raise ValueError(
        "No healthy data selection provided. Set HEALTHY_FLAG_COLUMN or both "
        "HEALTHY_START_SAMPLE and HEALTHY_END_SAMPLE."
    )

healthy = features[features["verified_healthy"]].copy()
if len(healthy) < 30:
    raise ValueError(f"Only {len(healthy)} verified healthy windows. Collect more healthy data (at least 30; more is preferable).")

print("Verified healthy windows:", len(healthy))
display(healthy[FEATURE_COLUMNS].describe())

## 6. Split healthy data into model-fit and threshold-calibration sets

The Isolation Forest is fitted on one part of the healthy data. A separate held-out healthy portion sets the anomaly threshold. This reduces the risk of calibrating the threshold on the exact same windows used to fit the model.

In [ ]:
from sklearn.model_selection import train_test_split

fit_df, calibration_df = train_test_split(
    healthy, test_size=0.25, random_state=RANDOM_STATE, shuffle=True
)

scaler = StandardScaler()
X_fit = scaler.fit_transform(fit_df[FEATURE_COLUMNS])
X_cal = scaler.transform(calibration_df[FEATURE_COLUMNS])

model = IsolationForest(
    n_estimators=N_ESTIMATORS,
    contamination=CONTAMINATION,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
model.fit(X_fit)

calibration_scores = model.decision_function(X_cal)  # lower = more anomalous
threshold = float(np.quantile(calibration_scores, HEALTHY_THRESHOLD_QUANTILE))

print("Fit windows:", len(fit_df))
print("Calibration windows:", len(calibration_df))
print("Anomaly threshold:", threshold)
print("Scores below threshold are flagged ABNORMAL.")

## 7. Inspect calibration scores and threshold

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(calibration_scores, bins=40)
plt.axvline(threshold, linestyle="--", label=f"Threshold = {threshold:.4f}")
plt.xlabel("Isolation Forest decision_function score")
plt.ylabel("Window count")
plt.title(f"{DEVICE_ID} — healthy calibration score distribution")
plt.legend()
plt.show()

## 8. Score all feature windows from this recording

In [ ]:
X_all = scaler.transform(features[FEATURE_COLUMNS])
features["anomaly_score"] = model.decision_function(X_all)
features["prediction"] = np.where(features["anomaly_score"] < threshold, "ABNORMAL", "NORMAL")

display(features[["device_id", "start_time_s", "end_time_s", "anomaly_score", "prediction"]].head(20))
print(features["prediction"].value_counts())

## 9. Optional evaluation on separately labelled data

For a meaningful evaluation, provide a separate real recording containing independently labelled healthy and abnormal windows. This notebook computes the same features from that raw file. The labels must be supplied by you; the model does not infer ground truth.

In [ ]:
# Optional: set to a second real raw CSV with a label column.
# Label values must be 'healthy' or 'abnormal'. Leave as None to skip.
TEST_RAW_CSV_PATH = None
TEST_LABEL_COLUMN = "label"

if TEST_RAW_CSV_PATH is not None:
    test_raw = pd.read_csv(TEST_RAW_CSV_PATH)
    for c in required:
        if c not in test_raw.columns:
            raise ValueError(f"Test file missing column {c}")
        test_raw[c] = pd.to_numeric(test_raw[c], errors="coerce")
    test_raw = test_raw.dropna(subset=required).reset_index(drop=True)

    test_signal = select_signal(test_raw)
    test_temp = test_raw[TEMPERATURE_COLUMN].to_numpy(dtype=float)
    test_rows = []
    for start in range(0, len(test_raw) - WINDOW_SIZE + 1, step):
        end = start + WINDOW_SIZE
        item = calculate_window_features(test_signal[start:end], test_temp[start:end])
        item["start_sample"], item["end_sample"] = start, end
        # Label window only when all raw labels in the window agree.
        if TEST_LABEL_COLUMN not in test_raw.columns:
            raise ValueError(f"Test file needs label column '{TEST_LABEL_COLUMN}'")
        labels = test_raw[TEST_LABEL_COLUMN].iloc[start:end].astype(str).str.lower().unique()
        item["label"] = labels[0] if len(labels) == 1 else "mixed"
        test_rows.append(item)

    test_features = pd.DataFrame(test_rows).dropna(subset=FEATURE_COLUMNS)
    test_features = test_features[test_features["label"].isin(["healthy", "abnormal"])]
    if len(test_features):
        X_test = scaler.transform(test_features[FEATURE_COLUMNS])
        test_features["prediction"] = np.where(
            model.decision_function(X_test) < threshold, "abnormal", "healthy"
        )
        print("Accuracy:", accuracy_score(test_features["label"], test_features["prediction"]))
        print(classification_report(test_features["label"], test_features["prediction"], zero_division=0))
        print(confusion_matrix(test_features["label"], test_features["prediction"],
                               labels=["healthy", "abnormal"]))
    else:
        print("No fully labelled test windows available.")

## 10. Save this motor's personalised profile

The profile stores the device ID, configuration, feature order, scaler, Isolation Forest, and calibrated threshold. Keep a separate profile for every motor. The device ID is metadata and a profile key; it is not itself an ML feature.

In [ ]:
profile = {
    "device_id": DEVICE_ID,
    "device_type": DEVICE_TYPE,
    "feature_columns": FEATURE_COLUMNS,
    "scaler": scaler,
    "model": model,
    "threshold": threshold,
    "config": {
        "sampling_rate_hz": SAMPLING_RATE_HZ,
        "window_size": WINDOW_SIZE,
        "overlap": OVERLAP,
        "signal_mode": SIGNAL_MODE,
        "accel_scale": ACCEL_SCALE,
        "temperature_column": TEMPERATURE_COLUMN,
        "accelerometer_columns": [ACCEL_X_COLUMN, ACCEL_Y_COLUMN, ACCEL_Z_COLUMN],
        "healthy_threshold_quantile": HEALTHY_THRESHOLD_QUANTILE
    },
    "training_counts": {
        "verified_healthy_windows": int(len(healthy)),
        "fit_windows": int(len(fit_df)),
        "calibration_windows": int(len(calibration_df))
    }
}

profile_path = os.path.join(OUTPUT_DIR, f"{DEVICE_ID}_profile.pkl")
joblib.dump(profile, profile_path)

feature_csv_path = os.path.join(OUTPUT_DIR, f"{DEVICE_ID}_calculated_features.csv")
features.to_csv(feature_csv_path, index=False)

print("Saved profile:", profile_path)
print("Saved calculated features:", feature_csv_path)

## 11. Load the profile and predict on new raw readings

Use exactly the same signal mode, acceleration scaling, sampling rate, window size, and feature calculations used during training.

In [ ]:
loaded = joblib.load(profile_path)

def predict_feature_windows(feature_frame, saved_profile):
    cols = saved_profile["feature_columns"]
    X = saved_profile["scaler"].transform(feature_frame[cols])
    scores = saved_profile["model"].decision_function(X)
    result = feature_frame.copy()
    result["anomaly_score"] = scores
    result["prediction"] = np.where(scores < saved_profile["threshold"], "ABNORMAL", "NORMAL")
    return result

# Demonstration on the feature windows calculated above (not new data):
prediction_check = predict_feature_windows(features, loaded)
display(prediction_check[["device_id", "anomaly_score", "prediction"]].head())

## 12. Deployment and interpretation notes

- This notebook calculates features from **raw sensor samples**; it does not expect precomputed RMS/peak/kurtosis columns.
- Each physical motor must have its own `DEVICE_ID` and independently trained profile.
- `NORMAL` means the window is sufficiently similar to this motor's learned healthy behaviour. `ABNORMAL` means it is unusual; Isolation Forest does **not** identify a named fault such as misalignment or bearing damage.
- A 1% healthy calibration quantile is a chosen operating point, not a guarantee of 1% field false alarms.
- Do not automatically retrain on every new reading. Add data to the healthy training history only after it has been independently verified as healthy, otherwise faults may become part of the baseline.
- The saved scikit-learn profile is intended for Python/backend inference. It is not directly loadable on the ESP32.
- Confirm that the live ESP32 feature-extraction implementation exactly matches this notebook before deployment.
- If using accelerometer magnitude, account for gravity and mounting orientation consistently. For benchmark comparability, note that a single-axis training dataset is not automatically equivalent to three-axis magnitude data.